# CSV Chunker / Heading-Aware, Size-Controlled Chunks

This notebook creates chunk-level CSV files from the document-level CSVs under `csvs/raw/`.

Outputs are written under:

- `csvs/chunked/policy/`
- `csvs/chunked/sentiment/`
- `csvs/chunked/synthetic/sentiment/`
- `csvs/chunked/`

## Methodological justification

The corpus contains long, heterogeneous policy and sentiment documents. Full-document vectorization can collapse several themes into one representation. Therefore, this notebook uses **heading-aware, size-controlled chunking**.

The approach:

- keeps headings as context;
- splits long sections into chunks of similar size;
- merges very short text with neighbouring content;
- preserves metadata from the original document;
- creates comparable units for BERTopic, TF-IDF, embeddings, and original-vs-synthetic distance testing.

This supports interpretability because each chunk remains traceable to its source document, corpus, country, and source type.

In [1]:
from pathlib import Path
import re
import pandas as pd

In [3]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)

    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / "csvs").exists() and (p / "markdown").exists():
            return p

    raise FileNotFoundError("Project root not found. Run inside MSC_RESEARCH_PROJECT.")


PROJECT_ROOT = find_project_root()

CSVS_RAW = PROJECT_ROOT / "csvs" / "raw"
CSVS_CHUNKED = PROJECT_ROOT / "csvs" / "chunked"

POLICY_CSV = CSVS_RAW / "policy" / "policy_documents_full.csv"
SENTIMENT_CSV = CSVS_RAW / "sentiment" / "sentiment_documents_full.csv"
SYNTHETIC_CSV = CSVS_RAW / "synthetic" / "sentiment" / "synthetic_sentiment_documents_full.csv"

OUT_POLICY = CSVS_CHUNKED / "policy"
OUT_SENTIMENT = CSVS_CHUNKED / "sentiment"
OUT_SYNTHETIC = CSVS_CHUNKED / "synthetic" / "sentiment"

for folder in [OUT_POLICY, OUT_SENTIMENT, OUT_SYNTHETIC]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

print("\nInput folders:")
print("Raw CSV folder:", CSVS_RAW)
print("Chunked CSV folder:", CSVS_CHUNKED)

print("\nInput CSV files:")
print("Policy CSV:", POLICY_CSV)
print("Sentiment CSV:", SENTIMENT_CSV)
print("Synthetic sentiment CSV:", SYNTHETIC_CSV)

print("\nOutput folders:")
print("Output policy chunks:", OUT_POLICY)
print("Output sentiment chunks:", OUT_SENTIMENT)
print("Output synthetic sentiment chunks:", OUT_SYNTHETIC)

Project root: /home/nsirim/Github/mscdsa/msc

Input folders:
Raw CSV folder: /home/nsirim/Github/mscdsa/msc/csvs/raw
Chunked CSV folder: /home/nsirim/Github/mscdsa/msc/csvs/chunked

Input CSV files:
Policy CSV: /home/nsirim/Github/mscdsa/msc/csvs/raw/policy/policy_documents_full.csv
Sentiment CSV: /home/nsirim/Github/mscdsa/msc/csvs/raw/sentiment/sentiment_documents_full.csv
Synthetic sentiment CSV: /home/nsirim/Github/mscdsa/msc/csvs/raw/synthetic/sentiment/synthetic_sentiment_documents_full.csv

Output folders:
Output policy chunks: /home/nsirim/Github/mscdsa/msc/csvs/chunked/policy
Output sentiment chunks: /home/nsirim/Github/mscdsa/msc/csvs/chunked/sentiment
Output synthetic sentiment chunks: /home/nsirim/Github/mscdsa/msc/csvs/chunked/synthetic/sentiment


In [4]:
def load_csv(path):
    if not path.exists():
        print("Missing:", path)
        return pd.DataFrame()

    df = pd.read_csv(path)
    df["text"] = df["text"].fillna("").astype(str)
    return df


df_policy = load_csv(POLICY_CSV)
df_sentiment = load_csv(SENTIMENT_CSV)
df_synthetic = load_csv(SYNTHETIC_CSV)

print("Policy:", df_policy.shape)
print("Sentiment:", df_sentiment.shape)
print("Synthetic sentiment:", df_synthetic.shape)

Policy: (56, 11)
Sentiment: (17, 11)
Synthetic sentiment: (53, 11)


In [5]:
def words(text):
    return re.findall(r"\S+", str(text))


def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", str(text))
    return [p.strip() for p in parts if p.strip()]


def sectionize_markdown(text):
    sections = []
    current_heading = ""
    buffer = []

    for line in str(text).splitlines():
        if re.match(r"^#{1,6}\s+", line.strip()):
            if buffer:
                sections.append((current_heading, "\n".join(buffer).strip()))
                buffer = []

            current_heading = re.sub(r"^#{1,6}\s+", "", line.strip())
        else:
            buffer.append(line)

    if buffer:
        sections.append((current_heading, "\n".join(buffer).strip()))

    return [(heading, body) for heading, body in sections if body.strip()]

In [6]:
def make_chunks_from_section(section_text, target_words=350, overlap_words=50):
    sentences = split_sentences(section_text)

    chunks = []
    current = []

    for sentence in sentences:
        candidate = " ".join(current + [sentence])

        if len(words(candidate)) <= target_words:
            current.append(sentence)
        else:
            if current:
                previous_chunk = " ".join(current).strip()
                chunks.append(previous_chunk)

                tail = words(previous_chunk)[-overlap_words:] if overlap_words else []
                current = [" ".join(tail), sentence] if tail else [sentence]
            else:
                current = [sentence]

    if current:
        chunks.append(" ".join(current).strip())

    return chunks

def normalize_for_chunking(text):
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_document(row, target_words=350, overlap_words=50, min_words=80):
    text = normalize_for_chunking(row["text"])
    sections = sectionize_markdown(text)

    chunks = []
    pending = ""

    for heading, body in sections:
        section_chunks = make_chunks_from_section(
            body,
            target_words=target_words,
            overlap_words=overlap_words,
        )

        for chunk in section_chunks:
            chunk_text = f"{heading}\n\n{chunk}".strip() if heading else chunk.strip()

            if len(words(chunk_text)) < min_words:
                pending = (pending + "\n\n" + chunk_text).strip()
                continue

            if pending:
                chunk_text = (pending + "\n\n" + chunk_text).strip()
                pending = ""

            chunks.append((heading, chunk_text))

    if pending:
        chunks.append(("", pending))

    return chunks

In [7]:
def chunk_dataframe(df, target_words=350, overlap_words=50, min_words=80):
    rows = []

    for _, row in df.iterrows():
        doc_chunks = chunk_document(
            row,
            target_words=target_words,
            overlap_words=overlap_words,
            min_words=min_words,
        )

        for idx, (heading, text) in enumerate(doc_chunks, start=1):
            base = row.to_dict()
            base.pop("text", None)

            base.update(
                {
                    "chunk_id": f"{row['doc_id']}_chunk_{idx:04d}",
                    "chunk_index": idx,
                    "heading_context": heading,
                    "chunk_text": text,
                    "chunk_char_count": len(text),
                    "chunk_word_count": len(words(text)),
                }
            )

            rows.append(base)

    return pd.DataFrame(rows)

In [8]:
TARGET_WORDS = 350
OVERLAP_WORDS = 50
MIN_WORDS = 80

chunks_policy = chunk_dataframe(
    df_policy,
    target_words=TARGET_WORDS,
    overlap_words=OVERLAP_WORDS,
    min_words=MIN_WORDS,
)

chunks_sentiment = chunk_dataframe(
    df_sentiment,
    target_words=TARGET_WORDS,
    overlap_words=OVERLAP_WORDS,
    min_words=MIN_WORDS,
)

chunks_synthetic = chunk_dataframe(
    df_synthetic,
    target_words=TARGET_WORDS,
    overlap_words=OVERLAP_WORDS,
    min_words=MIN_WORDS,
)

print("Policy chunks:", chunks_policy.shape)
print("Sentiment chunks:", chunks_sentiment.shape)
print("Synthetic sentiment chunks:", chunks_synthetic.shape)

Policy chunks: (2052, 16)
Sentiment chunks: (536, 16)
Synthetic sentiment chunks: (212, 16)


In [9]:
policy_out = OUT_POLICY / "policy_chunks.csv"
sentiment_out = OUT_SENTIMENT / "sentiment_chunks.csv"
synthetic_out = OUT_SYNTHETIC / "synthetic_sentiment_chunks.csv"

chunks_policy.to_csv(policy_out, index=False, encoding="utf-8")
chunks_sentiment.to_csv(sentiment_out, index=False, encoding="utf-8")
chunks_synthetic.to_csv(synthetic_out, index=False, encoding="utf-8")

chunks_all = pd.concat(
    [
        chunks_policy,
        chunks_sentiment,
        chunks_synthetic,
    ],
    ignore_index=True,
)

all_out = CSVS_CHUNKED / "chunks_all.csv"
chunks_all.to_csv(all_out, index=False, encoding="utf-8")

print("Saved:")
print(policy_out.relative_to(PROJECT_ROOT))
print(sentiment_out.relative_to(PROJECT_ROOT))
print(synthetic_out.relative_to(PROJECT_ROOT))
print(all_out.relative_to(PROJECT_ROOT))

Saved:
csvs/chunked/policy/policy_chunks.csv
csvs/chunked/sentiment/sentiment_chunks.csv
csvs/chunked/synthetic/sentiment/synthetic_sentiment_chunks.csv
csvs/chunked/chunks_all.csv


In [10]:
summary = (
    chunks_all
    .groupby(
        [
            "corpus",
            "source_type",
            "synthetic_type",
        ],
        dropna=False,
    )
    .agg(
        chunks=("chunk_id", "count"),
        documents=("doc_id", "nunique"),
        total_words=("chunk_word_count", "sum"),
        avg_words=("chunk_word_count", "mean"),
        total_chars=("chunk_char_count", "sum"),
        avg_chars=("chunk_char_count", "mean"),
    )
    .reset_index()
)

summary_out = CSVS_CHUNKED / "chunks_summary.csv"
summary.to_csv(summary_out, index=False, encoding="utf-8")

summary

,corpus,source_type,synthetic_type,chunks,documents,total_words,avg_words,total_chars,avg_chars
0,policy,original,NaN,2052,56,514694,250.825536,3549030,1729.546784
1,sentiment,original,NaN,536,17,138115,257.677239,912510,1702.444030
2,sentiment,synthetic,NaN,212,53,66406,313.235849,446667,2106.919811


In [11]:
preview_cols = [
    "chunk_id",
    "doc_id",
    "corpus",
    "source_type",
    "synthetic_type",
    "country",
    "heading_context",
    "chunk_word_count",
    "chunk_char_count",
]

chunks_all[preview_cols].head(20)

,chunk_id,doc_id,corpus,source_type,synthetic_type,country,heading_context,chunk_word_count,chunk_char_count
0,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,UNESCO - a global leader in education,80,547
1,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,The Global Education 2030 Agenda,275,1972
2,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Preparing students to be responsible and creat...,163,1172
3,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Foreword,347,2247
4,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Foreword,86,598
5,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Acknowledgements,328,2213
6,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Acknowledgements,142,957
7,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Table of contents,353,1498
8,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Table of contents,353,789
9,policy_ai_competency_framework_for_students_ch...,policy_ai_competency_framework_for_students,policy,original,NaN,NaN,Table of contents,353,857


In [12]:
sample_idx = 0

print(chunks_all.loc[sample_idx, "chunk_text"][:2500])

UNESCO - a global leader in education

Education is UNESCO's top priority because it is a basic human right and the foundation for peace and sustainable development. UNESCO is the United Nations' specialized agency for education, providing global and regional leadership to drive progress, strengthening the resilience and capacity of national systems to serve all learners. UNESCO also leads efforts to respond to contemporary global challenges through transformative learning, with special focus on gender equality and Africa across all actions.
